# Exploración AML: Física de Balances y Topología de Redes
En este notebook analizaremos datos transaccionales para detectar patrones de lavado de dinero. Combinaremos dos enfoques:
1. **Física de balances:** Anomalías a nivel contable (ej. vaciados de cuenta).
2. **Topología de red (Grafos):** Comportamiento estructural (ej. cuentas mula que concentran o dispersan fondos).

In [6]:
import duckdb
import networkx as nx
import pandas as pd

print("Leyendo y filtrando el CSV original...")
conn = duckdb.connect()

# Agregamos ../ para retroceder a la raíz del proyecto
df = conn.execute("""
    SELECT * 
    FROM read_csv_auto('../data/raw/paysim.csv')
    WHERE type IN ('TRANSFER', 'CASH_OUT')
""").df()

print(f"Transacciones listas para analizar: {len(df)}")
display(df.head())

Leyendo y filtrando el CSV original...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Transacciones listas para analizar: 2770409


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,TRANSFER,181.00,C1305486145,181.0,0.0,C553264065,0.0,0.00,1,0
1,1,CASH_OUT,181.00,C840083671,181.0,0.0,C38997010,21182.0,0.00,1,0
2,1,CASH_OUT,229133.94,C905080434,15325.0,0.0,C476402209,5083.0,51513.44,0,0
3,1,TRANSFER,215310.30,C1670993182,705.0,0.0,C1100439041,22425.0,0.00,0,0
4,1,TRANSFER,311685.89,C1984094095,10835.0,0.0,C932583850,6267.0,2719172.89,0,0


## 1. Análisis Transaccional: El Sensor de Vaciado
El primer paso es buscar anomalías contables básicas. Los estafadores suelen extraer el 100% de los fondos disponibles rápidamente para no dejar rastro. Mediremos cuántos fraudes cumplen con esta regla estricta.

In [7]:
# Filtramos solo los casos de fraude comprobado
df_fraud = df[df['isFraud'] == 1].copy()

# Creamos nuestro primer sensor lógico (flag_vaciado)
df_fraud['flag_vaciado'] = (df_fraud['amount'] == df_fraud['oldbalanceOrg']).astype(int)

vaciados = df_fraud['flag_vaciado'].sum()
total_fraudes = len(df_fraud)

print(f"De {total_fraudes} fraudes, {vaciados} vaciaron la cuenta origen a cero.")
display(df_fraud[['amount', 'oldbalanceOrg', 'newbalanceOrig', 'flag_vaciado']].head())

De 8213 fraudes, 8034 vaciaron la cuenta origen a cero.


,amount,oldbalanceOrg,newbalanceOrig,flag_vaciado
0,181.0,181.0,0.0,1
1,181.0,181.0,0.0,1
123,2806.0,2806.0,0.0,1
124,2806.0,2806.0,0.0,1
192,20128.0,20128.0,0.0,1


Los datos lo confirman: más de 8,000 casos fraudulentos dejan el saldo en cero.

## 2. Análisis de Red: Estructuras de Layering
Ahora pasamos a la estructura. En un esquema de lavado, el dinero fluye a través de intermediarios. Vamos a calcular:
* **In-degree (Fan-in):** De cuántas cuentas distintas recibe dinero un solo destino (posible mula recolectora).
* **Out-degree (Fan-out):** A cuántas cuentas distintas envía dinero un solo origen (posible mula dispersora).

In [8]:
# Contamos de cuántas cuentas distintas recibe dinero un destino (in-degree)
fan_in = df.groupby('nameDest')['nameOrig'].nunique().reset_index(name='in_degree')

# Contamos a cuántas cuentas distintas envía dinero un origen (out-degree)
fan_out = df.groupby('nameOrig')['nameDest'].nunique().reset_index(name='out_degree')

print("Top 5 cuentas destino (Posibles mulas colectoras - alto in-degree):")
display(fan_in.sort_values('in_degree', ascending=False).head())

print("\nTop 5 cuentas origen (Posibles dispersores - alto out-degree):")
display(fan_out.sort_values('out_degree', ascending=False).head())

Top 5 cuentas destino (Posibles mulas colectoras - alto in-degree):


,nameDest,in_degree
75478,C1286084959,75
421126,C665576141,68
95247,C1360767589,68
503591,C97730845,67
311527,C248609774,64



Top 5 cuentas origen (Posibles dispersores - alto out-degree):


,nameOrig,out_degree
1291341,C1902386530,3
1572805,C2098525306,3
2372816,C724452879,3
69019,C1048374464,2
422324,C129588043,2


## 3. Integración de Features Topológicas
Observamos cuentas destino (estanques colectores) recibiendo transferencias de hasta 75 orígenes distintos. El siguiente paso es unir estas métricas de red (`in_degree`, `out_degree`) al dataset principal para evaluar si el fraude estadísticamente tiene una mayor densidad de conexiones que el tráfico legítimo.

In [9]:
# Unimos las features topológicas al dataset original
df = df.merge(fan_in, on='nameDest', how='left')
df = df.merge(fan_out, on='nameOrig', how='left')

# Llenamos con 0 los casos donde no hay conexiones adicionales
df['in_degree'] = df['in_degree'].fillna(0)
df['out_degree'] = df['out_degree'].fillna(0)

# Comparamos el comportamiento de red entre operaciones normales y lavado de dinero
analisis_red = df.groupby('isFraud')[['in_degree', 'out_degree']].mean().round(2)

print("Promedio de conexiones (Legítimo vs Fraude):")
display(analisis_red)

Promedio de conexiones (Legítimo vs Fraude):


,in_degree,out_degree
isFraud,,
0,11.6,1.0
1,5.7,1.0


## 4. Cruce de Señales (Física + Red)
Descubrimiento clave: El tráfico legítimo tiene mayor `in_degree` promedio porque representa a comercios. El fraude opera en clusters más pequeños. Para atrapar el lavado, debemos cruzar el sensor de "vaciado de cuenta" con las métricas de red.

In [10]:
# Aplicamos el sensor de vaciado a TODO el dataset (no solo a los fraudes como antes)
df['flag_vaciado'] = (df['amount'] == df['oldbalanceOrg']).astype(int)

# Agrupamos por si es fraude y si hubo vaciado, viendo el comportamiento de la red
analisis_combinado = df.groupby(['isFraud', 'flag_vaciado'])['in_degree'].agg(['count', 'mean']).round(2)
analisis_combinado.columns = ['cantidad_transacciones', 'in_degree_promedio']

print("Intersección de Sensores: Vaciado x Topología")
display(analisis_combinado)

Intersección de Sensores: Vaciado x Topología


cantidad_transacciones  in_degree_promedio
isFraud flag_vaciado                                            
0       0                            2762196               11.60
1       0                                179                5.36
        1                               8034                5.71

## 5. Conclusión del EDA y Exportación
El análisis confirma nuestra hipótesis para el motor AML:
1. **El sensor de vaciado (`flag_vaciado`)** es un discriminador casi perfecto del fraude.
2. **Las métricas de red (`in_degree`, `out_degree`)** ayudan a diferenciar estafadores (que operan en pequeños clusters de ~5 conexiones) de pasarelas de pago legítimas (que reciben cientos de conexiones).

Guardaremos este dataset enriquecido con las variables topológicas y contables para entrenar el modelo XGBoost.

In [12]:
import os

# Aseguramos que la carpeta exista
os.makedirs('../data/processed', exist_ok=True)

# Guardamos el dataset enriquecido en formato Parquet
ruta_exportacion = '../data/processed/paysim_features.parquet'
df.to_parquet(ruta_exportacion, index=False)

print(f"Dataset con features de red guardado exitosamente en: {ruta_exportacion}")

Dataset con features de red guardado exitosamente en: ../data/processed/paysim_features.parquet
